# Forge scaled CAD GPU training
Private Kaggle notebook. Enable GPU and Internet; select Run All. Start with small, then increase to medium or large only after reviewing runtime, validation and quota. Three independent CUDA seeds. No language training. Source-level split, pinned CAD generator and GSO revision, per-object license screening. Weights are research-only. The notebook saves all evidence to Kaggle output; it does not publish a release or automatically upload to GitHub.

In [ ]:
import sys, pathlib, subprocess, datetime, urllib.request, torch
assert torch.cuda.is_available(), "Enable a GPU accelerator in Kaggle settings"
print("GPU:", torch.cuda.get_device_name(0))
ROOT=pathlib.Path("/kaggle/working/forge-cad-"+datetime.datetime.now(datetime.timezone.utc).strftime("%Y%m%dT%H%M%SZ"))
ROOT.mkdir()
SOURCE="15ced195ab932fc71dcd1c362a76887a6c4c9be2"
CAD="c2f050a54418d69a8ddc082d10110fb4d55e985d"
for name in ("cad_shape_v1.py","scaled_cad_v2.py"):
    urllib.request.urlretrieve(f"https://raw.githubusercontent.com/tpylman90-ctrl/Forge-Core/{SOURCE}/gpu-training/{name}", ROOT/name)
urllib.request.urlretrieve(f"https://raw.githubusercontent.com/tpylman90-ctrl/Native-model-forge/{CAD}/tools/generate_forge_basics.py",ROOT/"generate_forge_basics.py")
subprocess.run([sys.executable,"-m","pip","install","-q","trimesh","huggingface_hub"],check=True)


In [ ]:
# Start medium only after the small preset has demonstrated a working pipeline.
# Change PRESET to "medium" or "large" for a subsequent separate notebook run.
PRESET="small"
subprocess.run([sys.executable,str(ROOT/"scaled_cad_v2.py"),"--generator",str(ROOT/"generate_forge_basics.py"),"--out",str(ROOT/"results"),"--preset",PRESET,"--epochs","8","--batch","32"],check=True)


In [ ]:
import json, hashlib, zipfile
results=ROOT/"results"
rows=json.loads((results/"summary.json").read_text())
assert sorted(r["seed"] for r in rows)==[43,53,73]
manifest=json.loads((results/"dataset_manifest.json").read_text())
print("Actual source objects:",manifest["source_object_count"],"Actual samples:",manifest["sample_count"])
for row in rows: print(row["seed"],row["test"],"Gate:",row["gate"])
bundle=ROOT/"forge-cad-gpu-evidence.zip"
with zipfile.ZipFile(bundle,"w",zipfile.ZIP_DEFLATED) as z:
    for name in ("dataset_manifest.json","summary.json"): z.write(results/name,name)
    for row in rows:
        folder=results/f"seed-{row['seed']}"
        for name in ("metrics.json","curve.json","selected.pt","latest.pt"):
            f=folder/name
            if name=="selected.pt": assert hashlib.sha256(f.read_bytes()).hexdigest()==row["checkpoint_sha256"]
            z.write(f,f"{folder.name}/{name}")
    for name in ("cad_shape_v1.py","scaled_cad_v2.py","generate_forge_basics.py"): z.write(ROOT/name,name)
print("Saved all candidates and evidence:",bundle)
